# Độ phân tán theo seed trong MMKGC — thí nghiệm thăm dò

**Câu hỏi:** độ lệch chuẩn của MRR qua các seed có lớn hơn khoảng cách mà các bài MMKGC dùng để tuyên bố vượt SOTA hay không?

Mô hình khảo sát: **APKGC** (AAAI 2025). Chọn mô hình này vì nhẹ — trên máy Apple Silicon đo được khoảng 8 giây mỗi epoch, so với 160 giây của AdaMF-MAT vốn có thêm mạng sinh đối kháng. Nhờ vậy năm lần chạy vừa một phiên Kaggle.

**Trước khi chạy:** Settings → Accelerator → **GPU T4 x2** (hoặc P100), bật **Internet**.

---

### Nguyên tắc

- Siêu tham số **đọc trực tiếp từ script của tác giả**, không tự đặt
- Chỉ thay đổi duy nhất `-seed`
- Mọi con số đối chiếu đều trích từ bảng trong bài báo gốc, ghi rõ trang
- Kết luận nói về **những gì được công bố**, không suy ra tác giả đã làm gì

## 1. Môi trường

In [ ]:
import torch, subprocess, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
print("torch:", torch.__version__)

GPU_OK = False
NEED_DOWNGRADE = False

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    sm = f"sm_{cap[0]}{cap[1]}"
    print(f"GPU: {torch.cuda.get_device_name()}  {sm}")
    print("torch ho tro:", torch.cuda.get_arch_list())

    # Doc phien ban thoi khong du: phai thu chay that mot phep nhan ma tran.
    try:
        _ = (torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")).sum().item()
        GPU_OK = True
        print("\nGPU chay duoc.")
    except Exception as e:
        print(f"\nGPU khong chay duoc: {type(e).__name__}")
        print(f"torch {torch.__version__} khong co kernel cho {sm}.")
        NEED_DOWNGRADE = True
else:
    print("Khong thay GPU.")

print()
if NEED_DOWNGRADE:
    print("=> O tiep theo se cai ban torch co ho tro GPU nay.")
elif not GPU_OK:
    print("=> Se chay bang CPU, cham hon dang ke.")


### Xử lý khi GPU không tương thích

Kaggle có thể cấp Tesla P100 (sm_60) trong khi bản PyTorch cài sẵn chỉ hỗ trợ từ sm_70. Ô dưới chỉ chạy khi gặp trường hợp đó.

In [ ]:
# Chi chay khi o tren phat hien GPU khong tuong thich.
#
# Kaggle doi khi cap Tesla P100 (sm_60) trong khi ban torch cai san chi ho tro
# tu sm_70 tro len. Ha ve ban torch co kernel cho sm_60.
#
# Khong can khoi dong lai kernel: phan huan luyen duoc goi qua subprocess nen
# tien trinh con se nap ban torch moi cai.

if NEED_DOWNGRADE:
    print("Dang cai torch 2.4.1+cu121 (co kernel cho sm_60)...")
    # Go torchvision/torchaudio truoc de tranh xung dot rang buoc phien ban
    !pip uninstall -y -q torchvision torchaudio 2>&1 | tail -1
    !pip install -q --force-reinstall torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

    import subprocess, sys
    probe = (
        "import torch;"
        "a=torch.randn(8,8,device='cuda');"
        "print('torch',torch.__version__,'|',torch.cuda.get_device_name());"
        "print('matmul OK', float((a@a).sum()))"
    )
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    print(r.stdout or r.stderr[-500:])
    GPU_OK = "matmul OK" in r.stdout
    print("\nTien trinh con dung duoc GPU:", GPU_OK)
else:
    print("Khong can thay doi torch.")


## 2. Tải mã nguồn

In [ ]:
%cd /kaggle/working
!rm -rf APKGC _openke
!git clone -q https://github.com/HubuKG/APKGC.git
!ls APKGC

## 3. Chuẩn bị mã nguồn

Ô dưới xử lý năm vấn đề khiến repo không chạy được ngoài môi trường tác giả:

1. **Thiếu mã C++.** Repo chỉ có `Base.so` biên dịch sẵn cho Linux x86-64; thư mục `mmkgc/base/` chứa `Base.cpp` không được commit. `mmkgc/` là bản sửa của OpenKE nên lấy phần này từ đó.
2. Gọi `.cuda()` cứng ở 17 vị trí.
3. `requirement.txt` thiếu `scikit-learn`, `scipy`.
4. Kết quả ghi ra `../../data/` — ngoài thư mục dự án.
5. Sập khi chạy dưới 100 epoch vì `best_model_wts` được truy cập vô điều kiện.

In [ ]:
import os, glob, subprocess, sys

A = "/kaggle/working/APKGC"

# ---- 1. Bo sung ma nguon C++ con thieu -------------------------------------
# Repo APKGC chi commit ban Base.so bien dich san cho Linux x86-64, con thu muc
# mmkgc/base/ chua Base.cpp thi khong duoc commit -> khong the bien dich lai.
# mmkgc/ von la ban sua doi cua OpenKE (THUNLP) nen lay base/ tu do.
if not os.path.exists(f"{A}/mmkgc/base/Base.cpp"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/thunlp/OpenKE.git",
                    "/kaggle/working/_openke"], check=True)
    subprocess.run(["cp", "-r", "/kaggle/working/_openke/openke/base",
                    f"{A}/mmkgc/base"], check=True)
    print("Da bo sung mmkgc/base/ tu OpenKE")

# ---- 2. Module chon thiet bi ----------------------------------------------
DEVICE_SRC = "\n".join([
    "import os",
    "import torch",
    '_f = os.environ.get("MMKGC_DEVICE", "").strip().lower()',
    "if _f:",
    "    DEVICE = torch.device(_f)",
    "elif torch.cuda.is_available():",
    '    DEVICE = torch.device("cuda")',
    "else:",
    '    DEVICE = torch.device("cpu")',
    "def device_name():",
    "    return str(DEVICE)",
    "",
])
with open(f"{A}/mmkgc/device.py", "w") as f:
    f.write(DEVICE_SRC)

# ---- 3. Thay .cuda() cung bang thiet bi dong ------------------------------
total = 0
for p in glob.glob(f"{A}/**/*.py", recursive=True):
    if p.endswith("device.py"):
        continue
    s = open(p, encoding="utf-8").read()
    if ".cuda()" not in s:
        continue
    n = s.count(".cuda()")
    rel = os.path.relpath(p, A)
    if rel.startswith("mmkgc/"):
        depth = os.path.relpath(p, f"{A}/mmkgc").count(os.sep)
        imp = "from " + "." * (depth + 1) + "device import DEVICE"
    else:
        imp = "from mmkgc.device import DEVICE"
    if "device import DEVICE" not in s:
        lines = s.split("\n")
        last = max((i for i, l in enumerate(lines)
                    if l.startswith(("import ", "from "))), default=0)
        lines.insert(last + 1, imp)
        s = "\n".join(lines)
    s = s.replace(".cuda()", ".to(DEVICE)")
    open(p, "w", encoding="utf-8").write(s)
    total += n
    print(f"  {rel:<44} {n}")
print(f"Da thay {total} cho .cuda()")

# ---- 4. Sua cac loi khien khong chay duoc ---------------------------------
p = f"{A}/mmkgc/config/Trainer.py"
s = open(p, encoding="utf-8").read()
s = s.replace("data_loader=self.test_dataloader, use_gpu=True)",
              "data_loader=self.test_dataloader, use_gpu=(DEVICE.type != 'cpu'))")
if "self.best_model_wts = None" not in s:
    s = s.replace("        self.Loss_log = Loss_log(early_stop=args.early_stop)",
                  "        self.Loss_log = Loss_log(early_stop=args.early_stop)\n"
                  "        self.best_model_wts = None")
open(p, "w", encoding="utf-8").write(s)

p = f"{A}/args.py"
s = open(p, encoding="utf-8").read()
if "save_steps" not in s:
    s = s.replace("    arg.add_argument('-num_proj', type=int, default=1)",
                  "    arg.add_argument('-num_proj', type=int, default=1)\n"
                  "    arg.add_argument('-save_steps', type=int, default=100)")
open(p, "w", encoding="utf-8").write(s)

p = f"{A}/run_KGC_train.py"
s = open(p, encoding="utf-8").read()
s = s.replace("    torch.cuda.manual_seed_all(args.seed)",
              "    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)")
s = s.replace("    data_root = osp.abspath(osp.join(this_dir, '..', '..', 'data', ''))",
              "    data_root = osp.abspath(osp.join(this_dir, 'output'))")
s = s.replace("        save_steps=100,", "        save_steps=args.save_steps,")
s = s.replace("use_gpu=True", "use_gpu=(DEVICE.type != 'cpu')")
if "from mmkgc.device import DEVICE" not in s:
    s = s.replace("from args import get_args",
                  "from args import get_args\nfrom mmkgc.device import DEVICE, device_name")

OLD = "\n".join([
    '    save_dir = f"{args.save}-MRR{trainer.Loss_log.get_acc()}"',
    "    if not osp.exists(save_dir):",
    '        torch.save(trainer.best_model_wts, f"{args.save}-MRR{trainer.Loss_log.get_acc()}")',
    "",
    "    kge_score.load_checkpoint(save_dir)",
])
NEW = "\n".join([
    '    save_dir = f"{args.save}-MRR{trainer.Loss_log.get_acc()}"',
    "    if trainer.best_model_wts is not None:",
    "        if not osp.exists(save_dir):",
    "            torch.save(trainer.best_model_wts, save_dir)",
    "        kge_score.load_checkpoint(save_dir)",
])
if OLD in s:
    s = s.replace(OLD, NEW)
    print("Da sua phan luu checkpoint")
open(p, "w", encoding="utf-8").write(s)
print("Da sua Trainer/args/entrypoint")


### Biên dịch thư viện C++

In [ ]:
%cd /kaggle/working/APKGC/mmkgc
!bash make.sh 2>&1 | tail -3
!file release/Base.so
%cd /kaggle/working

### Cài thư viện phụ thuộc

In [ ]:
!pip install -q transformers easydict unidecode tensorboard scikit-learn scipy 2>&1 | tail -2
import transformers
from transformers.activations import ACT2FN
from transformers.pytorch_utils import apply_chunking_to_forward
print('transformers', transformers.__version__, '— cac ham can thiet OK')

## 4. Tải bộ nhúng

Đặc trưng thị giác và văn bản đã trích sẵn, không nằm trong repo.

Nguồn: https://drive.google.com/drive/folders/1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB

In [ ]:
!pip install -q gdown
%cd /kaggle/working/APKGC
!gdown --folder 1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB -O ./emb_dl -q

import shutil, pathlib
pathlib.Path('embeddings').mkdir(exist_ok=True)
moved = 0
for f in pathlib.Path('emb_dl').rglob('*'):
    if f.suffix in ('.pth', '.json'):
        dst = pathlib.Path('embeddings') / f.name
        if not dst.exists():
            shutil.move(str(f), str(dst)); moved += 1
print('Da chuyen', moved, 'file')
!ls -la embeddings/

### Kiểm chứng số chiều

Bước này cũng là một quan sát cho nghiên cứu: hai bộ dữ liệu dùng encoder khác nhau nên số chiều thị giác chênh nhau hơn mười lần.

In [ ]:
# Doc so chieu qua subprocess: tien trinh chinh co the dang giu ban torch cu
# trong bo nho sau khi ha phien ban, nen torch.load() o day se hong.
import subprocess, sys

probe = """
import torch, os
print(f"{'file':<24}{'shape':>18}")
print('-' * 42)
for f in sorted(os.listdir('embeddings')):
    if f.endswith('.pth'):
        t = torch.load(f'embeddings/{f}', map_location='cpu', weights_only=False)
        print(f"{f:<24}{str(tuple(t.shape)):>18}")
"""
r = subprocess.run([sys.executable, "-c", probe],
                   cwd="/kaggle/working/APKGC", capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])


## 5. Bộ khung chạy thí nghiệm

In [ ]:
import subprocess, sys, os, re, json, time, statistics

A = "/kaggle/working/APKGC"
OUT = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)

# APKGC in ket qua qua logger:
#   mrr:0.3742,  mr:145.2,  hit10:0.5011, hit3:0.4038, hit1:0.3061
LOGGER_LINE = re.compile(
    r"mrr:\s*([\d.]+).*?mr:\s*([\d.]+).*?hit10:\s*([\d.]+)"
    r".*?hit3:\s*([\d.]+).*?hit1:\s*([\d.]+)")

# Du phong: dinh dang printf cua mmkgc/base/Test.h
METRIC_LINE = re.compile(
    r"^\s*(l|r|averaged)\(filter\):\s*"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)", re.MULTILINE)

SHELL_VAR = re.compile(r"^\s*([A-Z_][A-Z0-9_]*)=(\S+)\s*$", re.MULTILINE)


def read_author_config():
    """Doc sieu tham so tu scripts/run_base.sh — file ma run.sh thuc su goi."""
    path = f"{A}/scripts/run_base.sh"
    cfg = dict(SHELL_VAR.findall(open(path).read()))
    for k in ["EMB_DIM", "NUM_BATCH", "MARGIN", "LR", "NEG_NUM", "EPOCH"]:
        assert k in cfg, f"{path} thieu {k}"
    return cfg


def parse_metrics(text):
    hits = LOGGER_LINE.findall(text)
    if hits:
        m = hits[-1]
        return dict(mrr=float(m[0]), mr=float(m[1]),
                    hit10=float(m[2]), hit3=float(m[3]), hit1=float(m[4]))
    sides = {}
    for m in METRIC_LINE.finditer(text):
        sides[m.group(1)] = dict(
            mrr=float(m.group(2)), mr=float(m.group(3)),
            hit10=float(m.group(4)), hit3=float(m.group(5)),
            hit1=float(m.group(6)))
    if "averaged" in sides:
        return sides["averaged"]
    if "l" in sides and "r" in sides:
        return {k: (sides["l"][k] + sides["r"][k]) / 2
                for k in ("mrr", "mr", "hit10", "hit3", "hit1")}
    return None


# Hai tap du lieu dung chien luoc hop nhat khac nhau, theo dung run.sh goc:
#   dong 1: DB15K  num_proj=2  Mformer_hd_graph
#   dong 2: MKG-W  num_proj=1  Mformer_hd_mean
DATASET_ARGS = {
    "DB15K": dict(num_proj=2, joint_way="Mformer_hd_graph"),
    "MKG-W": dict(num_proj=1, joint_way="Mformer_hd_mean"),
}


def run_seed(dataset, seed, cfg, epoch=None, save_steps=100):
    n_epoch = epoch or int(cfg["EPOCH"])
    da = DATASET_ARGS[dataset]
    os.makedirs(f"{A}/output/mmkgc/checkpoint", exist_ok=True)

    cmd = [sys.executable, "run_KGC_train.py",
           f"-dataset={dataset}",
           f"-num_proj={da['num_proj']}",
           "-use_intermediate=0",
           f"-joint_way={da['joint_way']}",
           f"-batch_size={cfg['NUM_BATCH']}",
           f"-margin={cfg['MARGIN']}",
           f"-epoch={n_epoch}",
           f"-dim={cfg['EMB_DIM']}",
           f"-neg_num={cfg['NEG_NUM']}",
           f"-learning_rate={cfg['LR']}",
           f"-noise_update={cfg.get('NU', 'epoch')}",
           "-noise_ratio=0.2", "-mask_ratio=0.7",
           f"-use_pool={cfg.get('POOL', 1)}",
           f"-add_noise={cfg.get('NOISE', 1)}",
           "-num_hidden_layers=1", "-num_attention_heads=2",
           f"-seed={seed}", f"-save=seed{seed}",
           f"-save_steps={save_steps}", f"-exp_id=seed{seed}"]

    print(f"  seed {seed} ...", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, cwd=A, capture_output=True, text=True)
    dt = time.time() - t0

    log = f"{OUT}/{dataset}_seed{seed}.log"
    with open(log, "w") as f:
        f.write(r.stdout + "\n===STDERR===\n" + r.stderr)

    m = parse_metrics(r.stdout + r.stderr)
    if m is None:
        print(f"    that bai sau {dt/60:.1f} phut — xem {log}")
        print("    " + "\n    ".join((r.stdout + r.stderr).strip().splitlines()[-10:]))
        return None
    m["seed"] = seed
    m["seconds"] = round(dt, 1)
    print(f"    MRR={m['mrr']:.4f}  H@1={m['hit1']:.4f}  ({dt/60:.1f} phut)")
    return m


def summarize(runs, dataset, cfg, n_epoch):
    if len(runs) < 2:
        print("Khong du ket qua de tinh do phan tan.")
        return None

    print("\n" + "=" * 68)
    print(f"KET QUA — APKGC tren {dataset}, {len(runs)} seed")
    print("=" * 68)
    print(f"{'seed':>6} {'MRR':>9} {'H@1':>9} {'H@3':>9} {'H@10':>9} {'phut':>7}")
    for r in runs:
        print(f"{r['seed']:>6} {r['mrr']:>9.4f} {r['hit1']:>9.4f} "
              f"{r['hit3']:>9.4f} {r['hit10']:>9.4f} {r['seconds']/60:>7.1f}")

    print()
    stats = {}
    for k in ("mrr", "hit1", "hit3", "hit10"):
        v = [r[k] for r in runs]
        stats[k] = dict(mean=statistics.mean(v), std=statistics.stdev(v),
                        lo=min(v), hi=max(v), spread=max(v) - min(v))
        s = stats[k]
        print(f"{k.upper():>6}: {s['mean']:.4f} +/- {s['std']:.4f}"
              f"   [{s['lo']:.4f}, {s['hi']:.4f}]   bien do {s['spread']:.4f}")

    payload = dict(model="APKGC", dataset=dataset, epoch=n_epoch,
                   config_source="scripts/run_base.sh", config=cfg,
                   dataset_args=DATASET_ARGS[dataset], runs=runs, stats=stats)
    with open(f"{OUT}/{dataset}_summary.json", "w") as f:
        json.dump(payload, f, indent=2)
    print(f"\nDa luu {OUT}/{dataset}_summary.json")
    return stats


In [ ]:
# Moi con so duoi day doc truc tiep tu bang trong bai bao APKGC (AAAI 2025).
PUBLISHED = {
    "DB15K": dict(mrr=0.364, hit1=0.282, hit3=0.413, hit10=0.527),
    "MKG-W": dict(mrr=0.374, hit1=0.306, hit3=0.404, hit10=0.501),
}

GAPS = [
    dict(dataset="DB15K", label="APKGC vs NativE", gap=0.002,
         src="Bang 2, tr.15010 (0.364 vs 0.362)"),
    dict(dataset="MKG-W", label="APKGC vs NativE", gap=0.008,
         src="Bang 2, tr.15010 (0.374 vs 0.366)"),
    dict(dataset="MKG-W", label="Dong gop attention penalty", gap=0.014,
         src="Bang 4, tr.15011 (0.374 vs 0.360)"),
    dict(dataset="MKG-W", label="Dong gop noise sampling", gap=0.016,
         src="Bang 4, tr.15011 (0.374 vs 0.358)"),
]


def interpret(stats, dataset):
    std = stats["mrr"]["std"]
    mean = stats["mrr"]["mean"]
    pub = PUBLISHED[dataset]["mrr"]

    print("=" * 68)
    print("DIEN GIAI")
    print("=" * 68)
    print(f"So cong bo trong bai bao : MRR = {pub:.4f}")
    print(f"So do lai duoc           : MRR = {mean:.4f} +/- {std:.4f}")
    print(f"Chenh lech               : {mean - pub:+.4f}")
    print()
    print("Doi chieu do phan tan voi khoang cach giua cac phuong phap:")
    for g in GAPS:
        if g["dataset"] != dataset:
            continue
        v = "NAM TRONG nhieu seed" if std >= g["gap"] else "vuot nhieu seed"
        print(f"  {g['label']:<34} {g['gap']:.4f}  ->  {v}")
        print(f"    nguon: APKGC AAAI 2025, {g['src']}")

    print()
    if std > 0.005:
        print("KET LUAN: do phan tan lon hon phan lon khoang cach duoc cong bo.")
        print("Luan diem cua nghien cuu dung vung — nen tien hanh day du.")
    elif std < 0.002:
        print("KET LUAN: mo hinh on dinh giua cac seed. Cac cai tien cong bo")
        print("nhieu kha nang la that. Nen chuyen trong tam sang phan audit")
        print("tham so va tinh khong dong nhat cua encoder.")
    else:
        print("KET LUAN: chua dut khoat. Can them seed hoac them mo hinh.")

    print()
    print("PHAM VI KET LUAN")
    print("Ket qua do do phan tan cua lan chay nay voi cau hinh duoc cong bo.")
    print("No KHONG cho phep suy ra tac gia da chay bao nhieu lan hay thu bao")
    print("nhieu cau hinh: kho ma chi phan anh nhung gi duoc cong bo. Phat bieu")
    print("kiem chung duoc la 'bai bao khong bao cao do lech chuan', khong phai")
    print("'tac gia khong do do lech chuan'.")


### Đối chiếu cấu hình: bài báo, README và mã nguồn

Ba nguồn này không khớp nhau. Ô dưới in ra để thấy rõ.

In [ ]:
cfg = read_author_config()
print('Doc tu scripts/run_base.sh (file ma run.sh thuc su goi):')
for k, v in cfg.items():
    print(f'  {k:<12} = {v}')
print()
print('Bai bao (AAAI 2025, tr.15010, muc Parameter Settings):')
print('  dim=128, batch=2048, neg=64, lr=1e-4')
print()
print('README cua repo:')
print('  EMB_DIM=128, NUM_BATCH=2048, NEG_NUM=64  (khop bai bao)')
print()
print('=> batch va neg trong script duoc goi chi bang MOT NUA gia tri cong bo.')
print('   Thi nghiem nay dung dung file ma run.sh goi.')

## 6. Chạy thử một seed

Kiểm tra pipeline thông suốt trước khi chạy đầy đủ. Đặt `save_steps` nhỏ để khâu đánh giá được kích hoạt sớm.

In [ ]:
# Smoke test BO QUA. Lan truoc no ton 33 phut va cho thay 0,825 phut/epoch
# tren DB15K — con so do gio dung de dat gioi han epoch o o duoi.
print('bo qua smoke test — da do 0,825 phut/epoch o lan chay 25/09')


## 7. Chạy đầy đủ — 5 seed

Dùng đúng số epoch trong script tác giả (8000, có early stopping sau 10 lần đánh giá không cải thiện). Nếu sắp hết thời gian phiên, đặt `EPOCH_OVERRIDE`.

In [ ]:
# DB15K la bo du lieu THU HAI cua bai bao (Bang 2). Muc dich: hien tuong
# thoi phong do chon checkpoint tren tap test (+0,0054 MRR tren MKG-W, 5 lan
# chay) co lap lai tren bo du lieu khac khong. Neu co, phat bieu khong con
# gioi han o mot bo du lieu.
# DB15K la bo du lieu THU HAI cua bai (Bang 2). run.sh cua tac gia dung cung
# EPOCH=8000 + early-stopping nhu MKG-W. Nhung DB15K co ~3x so bo ba train
# (99K vs 34K) nen do duoc 0,825 phut/epoch — MKG-W chay ~900 epoch truoc khi
# dung, tuc DB15K se can ~12,4 gio, VUOT phien Kaggle 12 gio.
#
# Vi vay CHAN o 600 epoch (toi da ~8,3 gio + overhead), 1 seed, va danh gia
# moi 50 epoch de co 12 diem tren duong cong. MKG-W dat dinh o epoch 200-600
# nen 600 la du de thay dinh; neu MRR con tang o epoch 600 thi ket luan phai
# noi ro la "chua hoi tu".
DATASET = 'DB15K'
SEEDS = [42]
EPOCH_OVERRIDE = 600
SAVE_STEPS = 50

n_epoch = EPOCH_OVERRIDE or int(cfg['EPOCH'])
print(f'Mo hinh  : APKGC')
print(f'Du lieu  : {DATASET}  {DATASET_ARGS[DATASET]}')
print(f'Epoch    : {n_epoch}  (early stopping sau 10 lan khong cai thien)')
print(f'Seed     : {SEEDS}')
print()

runs = []
for sd in SEEDS:
    r = run_seed(DATASET, sd, cfg, epoch=EPOCH_OVERRIDE, save_steps=SAVE_STEPS)
    if r:
        runs.append(r)
        # summarize() chi ghi file khi >= 2 seed; voi 1 seed phai tu ghi.
        r['epoch_cap'] = EPOCH_OVERRIDE; r['save_steps'] = SAVE_STEPS
        json.dump(r, open(f"{OUT}/{DATASET}_seed{sd}.json", "w"), indent=2)
        print(f"  da ghi {OUT}/{DATASET}_seed{sd}.json")

stats = summarize(runs, DATASET, cfg, n_epoch)

## 8. Diễn giải

In [ ]:
if stats:
    interpret(stats, DATASET)

## 9. Kết quả đã lưu

In [ ]:
!ls -la /kaggle/working/results/
import json, glob
for f in sorted(glob.glob('/kaggle/working/results/*_summary.json')):
    print('\n' + f)
    print(json.dumps(json.load(open(f))['stats'], indent=2))

---

### Phạm vi kết luận

Thí nghiệm đo độ phân tán của **một mô hình** trên **một bộ dữ liệu** với cấu hình được công bố. Nó cho biết mức nhiễu do seed gây ra trong điều kiện đó.

Nó **không** cho phép kết luận tác giả đã chạy bao nhiêu lần hay thử bao nhiêu cấu hình. Kho mã chỉ phản ánh những gì được công bố. Phát biểu kiểm chứng được là *bài báo không báo cáo độ lệch chuẩn*, không phải *tác giả không đo độ lệch chuẩn*.